# Expert 02 — Descriptors
Descriptors power `@property`, `classmethod`, `staticmethod`, and ORM fields.

## 1. Descriptor Protocol

In [ ]:
# A descriptor implements __get__, __set__, and/or __delete__
# Data descriptor: has __set__ or __delete__ (takes priority over instance dict)
# Non-data descriptor: only __get__ (instance dict takes priority)

class Descriptor:
    def __set_name__(self, owner, name):
        self.name = name
        self.private = f'_{name}'

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # accessed on class
        print(f'__get__ called for {self.name}')
        return getattr(obj, self.private, None)

    def __set__(self, obj, value):
        print(f'__set__ called for {self.name} = {value}')
        setattr(obj, self.private, value)

    def __delete__(self, obj):
        print(f'__delete__ called for {self.name}')
        delattr(obj, self.private)

class MyClass:
    x = Descriptor()

obj = MyClass()
obj.x = 42
print(obj.x)
del obj.x

## 2. Validated Descriptor

In [ ]:
class Validated:
    """Reusable validated attribute descriptor."""

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = f'_{name}'

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private_name, self.default)

    def __set__(self, obj, value):
        self.validate(value)
        setattr(obj, self.private_name, value)

    def validate(self, value):
        pass

    @property
    def default(self):
        return None


class PositiveInt(Validated):
    def validate(self, value):
        if not isinstance(value, int):
            raise TypeError(f'{self.public_name} must be int, got {type(value).__name__}')
        if value <= 0:
            raise ValueError(f'{self.public_name} must be positive, got {value}')

class NonEmptyStr(Validated):
    def validate(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f'{self.public_name} must be a non-empty string')


class Product:
    name  = NonEmptyStr()
    price = PositiveInt()
    stock = PositiveInt()

    def __init__(self, name, price, stock):
        self.name  = name
        self.price = price
        self.stock = stock

    def __repr__(self):
        return f'Product({self.name!r}, price={self.price}, stock={self.stock})'

p = Product('Widget', 10, 100)
print(p)

for bad in [('', 10, 5), ('X', -1, 5), ('X', 10, 0)]:
    try:
        Product(*bad)
    except (TypeError, ValueError) as e:
        print(f'Error: {e}')

## 3. Lazy / Cached Property

In [ ]:
class lazy_property:
    """Non-data descriptor: computed once, cached in instance dict."""

    def __init__(self, func):
        self.func = func
        self.__doc__ = func.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        print(f'Computing {self.name}...')
        value = self.func(obj)
        # Store in instance dict — shadows the descriptor on next access
        obj.__dict__[self.name] = value
        return value


class DataProcessor:
    def __init__(self, data):
        self.data = data

    @lazy_property
    def processed(self):
        """Expensive computation — only done once."""
        import time; time.sleep(0.01)  # simulate work
        return sorted(set(self.data))

    @lazy_property
    def stats(self):
        return {'min': min(self.data), 'max': max(self.data), 'mean': sum(self.data)/len(self.data)}


dp = DataProcessor([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])
print(dp.processed)  # computed
print(dp.processed)  # cached — no 'Computing' message
print(dp.stats)

## 4. How @property Works Internally

In [ ]:
# Implement property from scratch
class myproperty:
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        self.__doc__ = doc or (fget.__doc__ if fget else None)

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.fget is None:
            raise AttributeError('unreadable attribute')
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError('can\'t set attribute')
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError('can\'t delete attribute')
        self.fdel(obj)

    def getter(self, fget): return type(self)(fget, self.fset, self.fdel, self.__doc__)
    def setter(self, fset): return type(self)(self.fget, fset, self.fdel, self.__doc__)
    def deleter(self, fdel): return type(self)(self.fget, self.fset, fdel, self.__doc__)


class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius

    @myproperty
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError('Temperature below absolute zero')
        self._celsius = value

    @myproperty
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

t = Temperature(100)
print(f'{t.celsius}°C = {t.fahrenheit}°F')
t.celsius = 0
print(f'{t.celsius}°C = {t.fahrenheit}°F')